# Lab 02 - Tiền xử lý dữ liệu WDI chung cho nhóm 15

Notebook này là **file Jupyter Notebook duy nhất** cho toàn bộ phần tiền xử lý dữ liệu của nhóm. Toàn bộ dữ liệu đầu ra dùng cho Tableau được tạo từ file gốc `WDIEXCEL.xlsx` thông qua pipeline trong notebook này.

Mục tiêu của notebook:

1. Đọc dữ liệu WDI gốc từ `WDIEXCEL.xlsx`.
2. Chuẩn hóa metadata quốc gia và indicator.
3. Làm sạch dữ liệu theo một quy tắc chung cho toàn nhóm.
4. Tạo một file sạch nền tảng `wdi_clean_base.csv`.
5. Từ file sạch nền tảng, trích ra 4 file dữ liệu theo mục tiêu để kết nối Tableau.

Notebook chỉ dùng Python và các thư viện thông dụng/phổ biến cho tiền xử lý. Phần đọc file Excel lớn được triển khai bằng thư viện chuẩn `zipfile` và `xml.etree.ElementTree` để tránh phụ thuộc vào engine Excel ngoài môi trường nộp bài.


## Đối chiếu với yêu cầu đồ án

Toàn bộ code tiền xử lý dữ liệu phải được tổ chức trong **một file Jupyter Notebook (`.ipynb`)** kèm theo bài nộp. Notebook này đáp ứng yêu cầu đó theo cách sau:

| Yêu cầu | Cách notebook đáp ứng |
|---|---|
| Dùng Python cho tiền xử lý dữ liệu | Toàn bộ pipeline đọc, lọc, làm sạch, nội suy và xuất CSV được viết bằng Python. |
| Code nằm trong một file `.ipynb` | Chỉ sử dụng file `notebook/Nhom15_WDI_TienXuLy_Chung.ipynb` cho phần tiền xử lý. |
| Dữ liệu sau tiền xử lý dùng cho Tableau | Notebook xuất các file CSV trong `data_output/` để connect trực tiếp vào Tableau. |
| Có thể truy vết nguồn dữ liệu | File sạch chung giữ `Country_Code`, `Indicator_Code`, `Indicator_Name`, `Objective`, `Unit`, `Year`, `Value`. |

Các notebook riêng theo từng thành viên/mục tiêu trước đây đã được gộp vào notebook chung này để tránh việc mỗi mục tiêu tự đọc và làm sạch dữ liệu theo một logic khác nhau.


## Cấu trúc file nguồn `WDIEXCEL.xlsx`

File gốc WDI có nhiều sheet và rất nhiều cột vì đây là dữ liệu thô tổng quát của World Bank. Notebook không chuyển nguyên xi toàn bộ file Excel sang CSV, mà chỉ dùng các phần cần thiết cho bài phân tích.

Các sheet chính được dùng:

| Sheet | Vai trò trong pipeline |
|---|---|
| `Data` | Chứa giá trị các indicator theo quốc gia và năm. Đây là nguồn chính để lấy dữ liệu định lượng. |
| `Country` | Chứa metadata quốc gia: mã quốc gia, tên quốc gia, khu vực, nhóm thu nhập. Dùng để loại các dòng tổng hợp không phải quốc gia/vùng lãnh thổ cụ thể. |
| `Series` | Chứa metadata indicator: mã indicator, tên indicator, chủ đề. Dùng để xác minh các indicator nhóm chọn có tồn tại trong WDI. |

Các sheet metadata khác trong file gốc không được đưa nguyên vẹn vào output vì không cần cho dashboard và sẽ làm dữ liệu đầu ra cồng kềnh. Thay vào đó, pipeline chỉ giữ các trường metadata cần thiết để phân tích và truy vết.


## Danh sách indicator sử dụng

Notebook lọc 16 indicator thuộc 4 mục tiêu phân tích:

| Mục tiêu | Indicator | Cột sạch | Ý nghĩa |
|---|---|---|---|
| Kinh tế | `NY.GDP.MKTP.CD` | `GDP_current_USD` | GDP hiện hành, đo quy mô nền kinh tế. |
| Kinh tế | `NY.GDP.PCAP.CD` | `GDP_per_capita_current_USD` | GDP bình quân đầu người, đo mức sống kinh tế. |
| Kinh tế | `NY.GDP.MKTP.KD.ZG` | `GDP_growth_annual_pct` | Tăng trưởng GDP hằng năm. |
| Kinh tế | `SI.POV.DDAY` | `Poverty_headcount_3usd_2021PPP_pct` | Tỷ lệ nghèo theo chuẩn 3 USD/ngày. |
| Giáo dục | `SE.PRM.NENR` | `Primary_Net_Enrollment` | Tỷ lệ nhập học ròng tiểu học. |
| Giáo dục | `SE.SEC.NENR` | `Secondary_Net_Enrollment` | Tỷ lệ nhập học ròng trung học. |
| Giáo dục | `SE.ADT.LITR.ZS` | `Adult_Literacy_Rate` | Tỷ lệ biết chữ người trưởng thành. |
| Giáo dục | `SE.XPD.TOTL.GD.ZS` | `Education_Expenditure_GDP` | Chi tiêu giáo dục theo % GDP. |
| Sức khỏe | `SP.DYN.LE00.IN` | `Life_expectancy` | Tuổi thọ trung bình khi sinh. |
| Sức khỏe | `SH.DYN.MORT` | `Under5_mortality` | Tử vong trẻ em dưới 5 tuổi. |
| Sức khỏe | `SH.XPD.CHEX.GD.ZS` | `Health_expenditure_pct_GDP` | Chi tiêu y tế theo % GDP. |
| Sức khỏe | `SP.POP.TOTL` | `Population` | Tổng dân số. |
| Môi trường | `EN.GHG.CO2.MT.CE.AR5` | `CO2_Emissions_Total_MtCO2e` | Tổng phát thải CO2. |
| Môi trường | `EN.GHG.CO2.PC.CE.AR5` | `CO2_Emissions_Per_Capita` | Phát thải CO2 bình quân đầu người. |
| Môi trường | `EG.FEC.RNEW.ZS` | `Renewable_Energy_Pct` | Tỷ trọng năng lượng tái tạo. |
| Môi trường | `AG.LND.FRST.ZS` | `Forest_Area_Pct` | Tỷ lệ diện tích rừng. |


## Quy tắc làm sạch dữ liệu

Pipeline áp dụng cùng một quy tắc làm sạch cho tất cả mục tiêu để đảm bảo tính nhất quán:

1. **Lọc quốc gia/vùng lãnh thổ hợp lệ:** chỉ giữ các dòng có `Region` trong sheet `Country`; loại các aggregate rows như `World`, nhóm thu nhập, khu vực tổng hợp.
2. **Lọc giai đoạn phân tích:** chỉ giữ các năm 2000-2023.
3. **Xác minh indicator:** indicator phải nằm trong danh sách nhóm chọn và được kiểm tra trong sheet `Series`.
4. **Xử lý chuỗi không có dữ liệu:** nếu một cặp `Country_Code` - `Indicator_Code` không có quan sát nào trong giai đoạn 2000-2023, loại khỏi file sạch.
5. **Xử lý khoảng thiếu dài:** nếu khoảng thiếu nội bộ dài hơn 2 năm liên tiếp, loại cặp country-indicator đó để tránh tạo chuỗi dữ liệu giả.
6. **Nội suy khoảng thiếu ngắn:** nếu khoảng thiếu nội bộ dài tối đa 2 năm, nội suy tuyến tính giữa hai điểm quan sát thật.
7. **Không ngoại suy:** không tự điền giá trị ở đầu hoặc cuối chuỗi thời gian nếu không có điểm neo quan sát thật.

Cột `Was_Interpolated` trong `wdi_clean_base.csv` đánh dấu giá trị nào được tạo bởi nội suy tuyến tính.


## Schema các file đầu ra

### File sạch chung

`data_output/wdi_clean_base.csv` có schema:

```text
Country_Name, Country_Code, Region, Income_Group, Year,
Objective, Objective_Label, Indicator_Code, Indicator_Name,
Clean_Name, Unit, Value, Was_Interpolated
```

### File theo mục tiêu

Các file theo mục tiêu có schema chung:

```text
Country_Name, Country_Code, Region, Income_Group, Year, <các cột indicator đã pivot>
```

Ví dụ `wdi_health.csv` có các cột:

```text
Country_Name, Country_Code, Region, Income_Group, Year,
Life_expectancy, Under5_mortality, Health_expenditure_pct_GDP, Population
```


## Pipeline tiền xử lý

Cell dưới đây chứa toàn bộ code tiền xử lý. Khi chạy cell này, notebook sẽ:

1. đọc file `WDIEXCEL.xlsx`;
2. tạo `wdi_clean_base.csv`;
3. tạo 4 file dữ liệu theo mục tiêu;
4. tạo 2 file kiểm tra phụ `wdi_indicator_audit.csv` và `wdi_cleaning_report.csv`.

Hai file kiểm tra phụ không bắt buộc cho Tableau, nhưng giúp nhóm giải thích được indicator nào đã dùng và mỗi chuỗi country-indicator được giữ, loại hoặc nội suy như thế nào.


In [ ]:

from __future__ import annotations

import csv
import math
import zipfile
from pathlib import Path
from xml.etree import ElementTree as ET

ROOT = Path.cwd()
if not (ROOT / "WDIEXCEL.xlsx").exists() and (ROOT.parent / "WDIEXCEL.xlsx").exists():
    ROOT = ROOT.parent

XLSX_PATH = ROOT / "WDIEXCEL.xlsx"
OUTPUT_DIR = ROOT / "data_output"
YEARS = [str(y) for y in range(2000, 2024)]

OBJECTIVES = {
    "economy": {
        "output": "wdi_economy.csv",
        "label": "Kinh tế",
        "indicators": {
            "NY.GDP.MKTP.CD": {
                "clean_name": "GDP_current_USD",
                "unit": "US$",
                "role": "Quy mô nền kinh tế"
            },
            "NY.GDP.PCAP.CD": {
                "clean_name": "GDP_per_capita_current_USD",
                "unit": "US$ per person",
                "role": "Mức sống bình quân"
            },
            "NY.GDP.MKTP.KD.ZG": {
                "clean_name": "GDP_growth_annual_pct",
                "unit": "%",
                "role": "Tốc độ tăng trưởng"
            },
            "SI.POV.DDAY": {
                "clean_name": "Poverty_headcount_3usd_2021PPP_pct",
                "unit": "%",
                "role": "Tỷ lệ nghèo"
            }
        }
    },
    "education": {
        "output": "wdi_education.csv",
        "label": "Giáo dục",
        "indicators": {
            "SE.PRM.NENR": {
                "clean_name": "Primary_Net_Enrollment",
                "unit": "%",
                "role": "Mức độ tiếp cận giáo dục tiểu học"
            },
            "SE.SEC.NENR": {
                "clean_name": "Secondary_Net_Enrollment",
                "unit": "%",
                "role": "Mức độ tiếp cận giáo dục trung học"
            },
            "SE.ADT.LITR.ZS": {
                "clean_name": "Adult_Literacy_Rate",
                "unit": "%",
                "role": "Kết quả giáo dục qua tỷ lệ biết chữ"
            },
            "SE.XPD.TOTL.GD.ZS": {
                "clean_name": "Education_Expenditure_GDP",
                "unit": "% of GDP",
                "role": "Mức đầu tư của chính phủ cho giáo dục"
            }
        }
    },
    "health": {
        "output": "wdi_health.csv",
        "label": "Sức khỏe và dân số",
        "indicators": {
            "SP.DYN.LE00.IN": {
                "clean_name": "Life_expectancy",
                "unit": "years",
                "role": "Tuổi thọ trung bình"
            },
            "SH.DYN.MORT": {
                "clean_name": "Under5_mortality",
                "unit": "per 1,000 live births",
                "role": "Tỷ lệ tử vong trẻ em dưới 5 tuổi"
            },
            "SH.XPD.CHEX.GD.ZS": {
                "clean_name": "Health_expenditure_pct_GDP",
                "unit": "% of GDP",
                "role": "Chi tiêu y tế"
            },
            "SP.POP.TOTL": {
                "clean_name": "Population",
                "unit": "people",
                "role": "Quy mô dân số"
            }
        }
    },
    "environment": {
        "output": "wdi_environment.csv",
        "label": "Môi trường",
        "indicators": {
            "EN.GHG.CO2.MT.CE.AR5": {
                "clean_name": "CO2_Emissions_Total_MtCO2e",
                "unit": "Mt CO2e",
                "role": "Quy mô phát thải CO2"
            },
            "EN.GHG.CO2.PC.CE.AR5": {
                "clean_name": "CO2_Emissions_Per_Capita",
                "unit": "t CO2e/capita",
                "role": "Phát thải CO2 bình quân đầu người"
            },
            "EG.FEC.RNEW.ZS": {
                "clean_name": "Renewable_Energy_Pct",
                "unit": "%",
                "role": "Tỷ trọng năng lượng tái tạo"
            },
            "AG.LND.FRST.ZS": {
                "clean_name": "Forest_Area_Pct",
                "unit": "% of land area",
                "role": "Tỷ lệ diện tích rừng"
            }
        }
    }
}
ALL_INDICATORS = {
    code: {**meta, "objective": objective, "objective_label": spec["label"]}
    for objective, spec in OBJECTIVES.items()
    for code, meta in spec["indicators"].items()
}

NS = {
    "main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
    "rel": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
    "pkgrel": "http://schemas.openxmlformats.org/package/2006/relationships",
}


def col_index(cell_ref: str) -> int:
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    value = 0
    for ch in letters:
        value = value * 26 + ord(ch.upper()) - 64
    return value - 1


def load_shared_strings(zf: zipfile.ZipFile) -> list[str]:
    if "xl/sharedStrings.xml" not in zf.namelist():
        return []
    values: list[str] = []
    with zf.open("xl/sharedStrings.xml") as fh:
        for _, elem in ET.iterparse(fh, events=("end",)):
            if elem.tag.endswith("}si"):
                texts = [node.text or "" for node in elem.iter() if node.tag.endswith("}t")]
                values.append("".join(texts))
                elem.clear()
    return values


def get_sheet_paths(zf: zipfile.ZipFile) -> dict[str, str]:
    workbook = ET.fromstring(zf.read("xl/workbook.xml"))
    rels = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
    rel_map = {rel.attrib["Id"]: rel.attrib["Target"] for rel in rels.findall("pkgrel:Relationship", NS)}
    paths: dict[str, str] = {}
    for sheet in workbook.findall("main:sheets/main:sheet", NS):
        name = sheet.attrib["name"]
        rel_id = sheet.attrib[f"{{{NS['rel']}}}id"]
        target = rel_map[rel_id].replace("\\", "/")
        if not target.startswith("xl/"):
            target = "xl/" + target
        paths[name] = target
    return paths


def cell_value(cell: ET.Element, shared: list[str]):
    cell_type = cell.attrib.get("t")
    if cell_type == "inlineStr":
        return "".join(node.text or "" for node in cell.iter() if node.tag.endswith("}t"))
    value = cell.find("main:v", NS)
    if value is None or value.text is None:
        return None
    raw = value.text
    if cell_type == "s":
        return shared[int(raw)]
    if cell_type == "b":
        return raw == "1"
    try:
        return float(raw)
    except ValueError:
        return raw


def iter_rows(zf: zipfile.ZipFile, sheet_path: str, shared: list[str]):
    with zf.open(sheet_path) as fh:
        for _, row in ET.iterparse(fh, events=("end",)):
            if not row.tag.endswith("}row"):
                continue
            values = {}
            current_index = -1
            for cell in row.findall("main:c", NS):
                ref = cell.attrib.get("r", "")
                index = col_index(ref) if ref else current_index + 1
                current_index = index
                values[index] = cell_value(cell, shared)
            if values:
                max_col = max(values)
                yield [values.get(i) for i in range(max_col + 1)]
            row.clear()


def read_small_sheet(zf: zipfile.ZipFile, path: str, shared: list[str]) -> list[dict[str, object]]:
    rows = iter_rows(zf, path, shared)
    header = [str(v) if v is not None else "" for v in next(rows)]
    out = []
    for row in rows:
        out.append({header[i]: row[i] if i < len(row) else None for i in range(len(header))})
    return out


def as_float(value):
    if value is None or value == "":
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    if math.isnan(number):
        return None
    return number


def longest_internal_gap(values: list[float | None]) -> int:
    valid = [idx for idx, value in enumerate(values) if value is not None]
    if not valid:
        return len(values)
    start, end = min(valid), max(valid)
    best = current = 0
    for value in values[start : end + 1]:
        if value is None:
            current += 1
            best = max(best, current)
        else:
            current = 0
    return best


def interpolate_short_gaps(values: list[float | None]) -> tuple[list[float | None], set[int]]:
    cleaned = values[:]
    filled_indices: set[int] = set()
    valid = [idx for idx, value in enumerate(cleaned) if value is not None]
    if len(valid) < 2:
        return cleaned, filled_indices
    for left, right in zip(valid, valid[1:]):
        gap = right - left - 1
        if 0 < gap <= 2:
            start, end = cleaned[left], cleaned[right]
            assert start is not None and end is not None
            step = (end - start) / (right - left)
            for offset in range(1, gap + 1):
                idx = left + offset
                cleaned[idx] = start + step * offset
                filled_indices.add(idx)
    return cleaned, filled_indices


def clean_country_indicator(values: list[float | None]) -> tuple[list[float | None] | None, str, set[int]]:
    observed = sum(value is not None for value in values)
    if observed == 0:
        return None, "dropped_no_observation", set()
    if longest_internal_gap(values) > 2:
        return None, "dropped_internal_gap_gt_2_years", set()
    cleaned, filled_indices = interpolate_short_gaps(values)
    return cleaned, "kept", filled_indices


def write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8-sig") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def build_base_dataset() -> tuple[list[dict], list[dict], list[dict]]:
    if not XLSX_PATH.exists():
        raise FileNotFoundError(f"Missing source file: {XLSX_PATH}")
    with zipfile.ZipFile(XLSX_PATH) as zf:
        shared = load_shared_strings(zf)
        paths = get_sheet_paths(zf)
        series_rows = read_small_sheet(zf, paths["Series"], shared)
        country_rows = read_small_sheet(zf, paths["Country"], shared)

        real_countries = {
            str(row["Country Code"]): {
                "Country_Name": row.get("Table Name") or row.get("Short Name") or row.get("Country Code"),
                "Country_Code": row.get("Country Code"),
                "Region": row.get("Region"),
                "Income_Group": row.get("Income Group"),
            }
            for row in country_rows
            if row.get("Region") not in (None, "")
        }
        series_by_code = {str(row["Series Code"]): row for row in series_rows}

        rows = iter_rows(zf, paths["Data"], shared)
        header = [str(v) if v is not None else "" for v in next(rows)]
        idx = {name: header.index(name) for name in ["Country Code", "Indicator Code"]}
        year_idx = {year: header.index(year) for year in YEARS if year in header}

        raw: dict[tuple[str, str], list[float | None]] = {}
        for row in rows:
            indicator_code = str(row[idx["Indicator Code"]]) if idx["Indicator Code"] < len(row) and row[idx["Indicator Code"]] is not None else ""
            if indicator_code not in ALL_INDICATORS:
                continue
            country_code = str(row[idx["Country Code"]]) if idx["Country Code"] < len(row) and row[idx["Country Code"]] is not None else ""
            if country_code not in real_countries:
                continue
            raw[(country_code, indicator_code)] = [as_float(row[year_idx[year]]) if year_idx[year] < len(row) else None for year in YEARS]

    audit_rows = []
    for code, meta in ALL_INDICATORS.items():
        series = series_by_code.get(code, {})
        audit_rows.append({
            "Objective": meta["objective"],
            "Objective_Label": meta["objective_label"],
            "Indicator_Code": code,
            "Clean_Name": meta["clean_name"],
            "Confirmed_In_WDI": "Yes" if code in series_by_code else "No",
            "Indicator_Name": series.get("Indicator Name", ""),
            "Topic": series.get("Topic", ""),
            "Unit": meta.get("unit", ""),
            "Analysis_Role": meta.get("role", ""),
        })

    base_rows = []
    cleaning_rows = []
    for country_code, country in sorted(real_countries.items()):
        for indicator_code, meta in ALL_INDICATORS.items():
            values = raw.get((country_code, indicator_code), [None] * len(YEARS))
            cleaned, status, filled_indices = clean_country_indicator(values)
            cleaning_rows.append({
                "Country_Code": country_code,
                "Country_Name": country["Country_Name"],
                "Objective": meta["objective"],
                "Indicator_Code": indicator_code,
                "Clean_Name": meta["clean_name"],
                "Observed_Values": sum(value is not None for value in values),
                "Internal_Max_Missing_Gap": longest_internal_gap(values),
                "Interpolated_Values": len(filled_indices),
                "Status": status,
            })
            if cleaned is None:
                continue
            series = series_by_code.get(indicator_code, {})
            for i, (year, value) in enumerate(zip(YEARS, cleaned)):
                if value is None:
                    continue
                base_rows.append({
                    **country,
                    "Year": int(year),
                    "Objective": meta["objective"],
                    "Objective_Label": meta["objective_label"],
                    "Indicator_Code": indicator_code,
                    "Indicator_Name": series.get("Indicator Name", ""),
                    "Clean_Name": meta["clean_name"],
                    "Unit": meta.get("unit", ""),
                    "Value": value,
                    "Was_Interpolated": "Yes" if i in filled_indices else "No",
                })
    base_rows.sort(key=lambda r: (r["Objective"], r["Country_Name"], r["Indicator_Code"], r["Year"]))
    return base_rows, audit_rows, cleaning_rows


def build_objective_outputs(base_rows: list[dict]) -> dict[str, int]:
    counts = {}
    for objective, spec in OBJECTIVES.items():
        metric_order = [meta["clean_name"] for meta in spec["indicators"].values()]
        wide_map: dict[tuple[str, int], dict] = {}
        for row in base_rows:
            if row["Objective"] != objective:
                continue
            key = (row["Country_Code"], int(row["Year"]))
            wide = wide_map.setdefault(key, {
                "Country_Name": row["Country_Name"],
                "Country_Code": row["Country_Code"],
                "Region": row["Region"],
                "Income_Group": row["Income_Group"],
                "Year": int(row["Year"]),
            })
            wide[row["Clean_Name"]] = row["Value"]
        wide_rows = sorted(wide_map.values(), key=lambda r: (str(r["Country_Name"]), int(r["Year"])))
        write_csv(OUTPUT_DIR / spec["output"], wide_rows, ["Country_Name", "Country_Code", "Region", "Income_Group", "Year", *metric_order])
        counts[spec["output"]] = len(wide_rows)
    return counts


def main() -> None:
    base_rows, audit_rows, cleaning_rows = build_base_dataset()
    write_csv(
        OUTPUT_DIR / "wdi_clean_base.csv",
        base_rows,
        ["Country_Name", "Country_Code", "Region", "Income_Group", "Year", "Objective", "Objective_Label", "Indicator_Code", "Indicator_Name", "Clean_Name", "Unit", "Value", "Was_Interpolated"],
    )
    write_csv(
        OUTPUT_DIR / "wdi_indicator_audit.csv",
        audit_rows,
        ["Objective", "Objective_Label", "Indicator_Code", "Clean_Name", "Confirmed_In_WDI", "Indicator_Name", "Topic", "Unit", "Analysis_Role"],
    )
    write_csv(
        OUTPUT_DIR / "wdi_cleaning_report.csv",
        cleaning_rows,
        ["Country_Code", "Country_Name", "Objective", "Indicator_Code", "Clean_Name", "Observed_Values", "Internal_Max_Missing_Gap", "Interpolated_Values", "Status"],
    )
    counts = build_objective_outputs(base_rows)
    print(f"Wrote data_output/wdi_clean_base.csv: {len(base_rows):,} long-format rows")
    for name, count in counts.items():
        print(f"Wrote data_output/{name}: {count:,} wide rows")
    print("Wrote data_output/wdi_indicator_audit.csv")
    print("Wrote data_output/wdi_cleaning_report.csv")

main()


## Kiểm tra nhanh output

Cell dưới đây dùng để kiểm tra sau khi chạy pipeline:

- các file CSV đã được tạo hay chưa;
- số dòng/cột của từng file;
- danh sách indicator trong file sạch chung;
- số lượng giá trị được nội suy.


In [ ]:
import pandas as pd
from pathlib import Path

root = Path.cwd()
if not (root / "data_output").exists() and (root.parent / "data_output").exists():
    root = root.parent

output_dir = root / "data_output"
files = [
    "wdi_clean_base.csv",
    "wdi_economy.csv",
    "wdi_education.csv",
    "wdi_health.csv",
    "wdi_environment.csv",
    "wdi_indicator_audit.csv",
    "wdi_cleaning_report.csv",
]

for file_name in files:
    path = output_dir / file_name
    df = pd.read_csv(path)
    print(f"{file_name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

base = pd.read_csv(output_dir / "wdi_clean_base.csv")
print("\nIndicators in clean base:")
print(
    base[["Objective", "Indicator_Code", "Clean_Name"]]
    .drop_duplicates()
    .sort_values(["Objective", "Indicator_Code"])
    .to_string(index=False)
)

print("\nInterpolated values:")
print(base["Was_Interpolated"].value_counts(dropna=False).to_string())


## Kết luận tiền xử lý

Sau tiền xử lý, nhóm có một nguồn dữ liệu sạch chung để đảm bảo nhất quán giữa các mục tiêu. Các file CSV theo mục tiêu được sinh từ cùng một nguồn này, nên dashboard Tableau có thể kết nối từng file riêng nhưng vẫn dựa trên cùng quy tắc lọc quốc gia, chọn năm, xử lý missing value và chuẩn hóa indicator.
